# 문장별 감성 점수 계산

처음에는 전체 셀을 실행한다. 이후에는 마지막 셀의 `sentense` 리스트에 문장을 추가하고 해당 셀만 다시 실행한다.

In [18]:
from pathlib import Path
import json
import os
import sys

# 프로젝트 루트와 notebooks/ 양쪽에서 같은 모듈과 사전을 사용한다.
root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "src" / "korean_sentiment.py").is_file())
module_dir = str(root / "src")
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)

# 커널이 셸의 JAVA_HOME을 상속하지 않은 경우 프로젝트의 JDK를 찾는다.
if not os.environ.get("JAVA_HOME"):
    manifest_path = root / "data" / "setup_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.is_file() else {}
    jdk_dir = root / "data" / f"jdk-{manifest.get('java_major', 25)}"
    candidates = [jdk_dir / "Contents" / "Home", jdk_dir]
    if manifest.get("java_home"):
        candidates.insert(0, Path(manifest["java_home"]))
    for candidate in candidates:
        if (candidate / "bin" / "java").is_file():
            os.environ["JAVA_HOME"] = str(candidate)
            break

from korean_sentiment import OktTokenizer, SentimentAnalyzer, load_assignment_lexicon, print_review_analysis

tokenizer = OktTokenizer()
analyzer = SentimentAnalyzer(load_assignment_lexicon(tokenizer))

In [21]:
sentense = [
    # "영화가 재미없진 않아",
    # "영화가 재미없는 건 아니야",
    # "영화가 노잼 아니야",
    # "영화가 노잼이야",
    # "영화 영 아니야",
    # "영화 좋아",
    # "영화 좋아 진짜 좋아 아주 많이",

    # 1. 같은 위치에서 짧은 표현과 긴 표현이 모두 일치할 때
    "결말이 좋다.",                 # +2
    "결말이 좋지 않다.",            # -2

    # 2. 같은 감성어에 앞쪽 부정어와 뒤쪽 부정어를 적용할 때
    "전개가 정말 뻔하다.",          # -1.5
    "전개가 정말 안 뻔하다.",       # +1.5
    "전개가 정말 뻔하지 않다.",     # +1.5

    # 3. 강조어 탐색이 문장부호나 다른 감성 표현에서 멈출 때
    "정말 명작.",                  # +3
    "정말, 명작.",                 # +2
    "정말 명작이지만 뻔하다.",      # +2
]

for sentence in sentense:
    tokens = tokenizer.tokenize(sentence)
    result = analyzer.score_tokens(tokens)
    print_review_analysis(sentence, tokens, result)
    print()

리뷰: 결말이 좋다.
형태소/품사: 결말/Noun 이/Josa 좋다/Adjective ./Punctuation
  좋거나 (KNU 원본): 기본 +2 × 부정 계수 1 (부정 0회) × 강도 1 (강조어 없음) = +2
리뷰 감성 점수: +2

리뷰: 결말이 좋지 않다.
형태소/품사: 결말/Noun 이/Josa 좋다/Adjective 않다/Verb ./Punctuation
  좋지 않거나 (KNU 원본): 기본 -2 × 부정 계수 1 (부정 0회) × 강도 1 (강조어 없음) = -2
리뷰 감성 점수: -2

리뷰: 전개가 정말 뻔하다.
형태소/품사: 전개/Noun 가/Josa 정말/Noun 뻔하다/Adjective ./Punctuation
  뻔하다 (과제 추가): 기본 -1 × 부정 계수 1 (부정 0회) × 강도 1.5 (정말) = -1.5
리뷰 감성 점수: -1.5

리뷰: 전개가 정말 안 뻔하다.
형태소/품사: 전개/Noun 가/Josa 정말/Noun 안/Noun 뻔하다/Adjective ./Punctuation
  뻔하다 (과제 추가): 기본 -1 × 부정 계수 -1 (부정 1회) × 강도 1.5 (정말) = +1.5
리뷰 감성 점수: +1.5

리뷰: 전개가 정말 뻔하지 않다.
형태소/품사: 전개/Noun 가/Josa 정말/Noun 뻔하다/Adjective 않다/Verb ./Punctuation
  뻔하다 (과제 추가): 기본 -1 × 부정 계수 -1 (부정 1회) × 강도 1.5 (정말) = +1.5
리뷰 감성 점수: +1.5

리뷰: 정말 명작.
형태소/품사: 정말/Noun 명작/Noun ./Punctuation
  명작 (과제 추가): 기본 +2 × 부정 계수 1 (부정 0회) × 강도 1.5 (정말) = +3
리뷰 감성 점수: +3

리뷰: 정말, 명작.
형태소/품사: 정말/Noun ,/Punctuation 명작/Noun ./Punctuation
  명작 (과제 추가): 기본 +2 × 부정 계수 1 (부정 0회) × 강도 1 (강조어 없음)